In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, accuracy_score, calibration_curve
from itertools import combinations

sns.set_theme(style='whitegrid', font_scale=1.05)
PALETTE    = sns.color_palette('tab10', 5)
STACK_COR  = '#e63946'

FEATURES = ['prob_svm', 'prob_knn', 'prob_extra_trees', 'prob_regressao_logistica', 'prob_lightgbm']
MODELOS  = ['svm', 'knn', 'extra_trees', 'regressao_logistica', 'lightgbm']
LABELS   = ['SVM', 'KNN', 'Extra Trees', 'Reg. Logística', 'LightGBM']
JANELAS  = [5, 10, 15]
TARGET   = 'Resultado Real'

BASE       = os.path.abspath(os.path.join('..'))
DIR_STACK  = os.path.join(BASE, 'results', 'experimento_04_stack')
DIR_DET    = os.path.join(BASE, 'results', 'experimento_03_detalhado')

## Carregamento e merge dos dados

In [ ]:
df_meta = pd.read_csv(os.path.join(DIR_STACK, 'meta_dataset_stack.csv'))

df_stack_pred = pd.read_csv(os.path.join(DIR_STACK, 'stack_ensemble_predicoes.csv'))
df_stack_pred = df_stack_pred.rename(columns={
    'Previsao':              'pred_stack',
    'Probabilidade Classe 1':'prob_stack'
})

df_all = df_meta.copy()

for m in MODELOS:
    path = os.path.join(DIR_DET, f'{m}_experimento_03_predicoes_detalhadas.csv')
    df_m = pd.read_csv(path)[['Janela Incremental', 'Temporada', 'Ordem Jogo Temporada', 'Previsao']]
    df_m = df_m.rename(columns={'Previsao': f'pred_{m}'})
    df_all = df_all.merge(df_m, on=['Janela Incremental', 'Temporada', 'Ordem Jogo Temporada'], how='left')

df_all = df_all.merge(
    df_stack_pred[['Janela Incremental', 'Temporada', 'Ordem Jogo Temporada', 'pred_stack', 'prob_stack']],
    on=['Janela Incremental', 'Temporada', 'Ordem Jogo Temporada'],
    how='left'
)

for m in MODELOS:
    df_all[f'erro_{m}'] = (df_all[f'pred_{m}'] != df_all[TARGET]).astype(float)
df_all['erro_stack'] = (df_all['pred_stack'] != df_all[TARGET]).astype(float)

print(f'Shape total: {df_all.shape}')
df_all.head(3)

## 1. Diversidade entre os modelos base

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, janela in zip(axes, JANELAS):
    cols_erro = [f'erro_{m}' for m in MODELOS]
    df_j = df_all[df_all['Janela Incremental'] == janela].dropna(subset=cols_erro)
    erros = df_j[cols_erro].copy()
    erros.columns = LABELS
    corr = erros.corr()

    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True

    sns.heatmap(
        corr, ax=ax, annot=True, fmt='.2f',
        cmap='coolwarm', vmin=-1, vmax=1,
        linewidths=0.5, square=True,
        mask=mask
    )
    ax.set_title(f'Janela = {janela}', fontsize=12)

fig.suptitle('Correlação dos Erros entre Modelos Base', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
records_div = []
for janela in JANELAS:
    cols_pred = [f'pred_{m}' for m in MODELOS]
    cols_erro = [f'erro_{m}' for m in MODELOS]
    df_j = df_all[df_all['Janela Incremental'] == janela].dropna(subset=cols_pred + cols_erro)

    for (i, (m1, l1)), (j, (m2, l2)) in combinations(enumerate(zip(MODELOS, LABELS)), 2):
        disagreement  = (df_j[f'pred_{m1}'] != df_j[f'pred_{m2}']).mean()
        double_fault  = ((df_j[f'erro_{m1}'] == 1) & (df_j[f'erro_{m2}'] == 1)).mean()
        records_div.append({
            'Janela': janela,
            'Par': f'{l1} / {l2}',
            'Disagreement': disagreement,
            'Double Fault': double_fault
        })

df_div = pd.DataFrame(records_div)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for ax, metric, title in zip(
    axes,
    ['Disagreement', 'Double Fault'],
    ['Disagreement entre pares (proporção de jogos em que discordam)',
     'Double Fault entre pares (ambos erram simultaneamente)']
):
    df_piv = df_div.pivot(index='Par', columns='Janela', values=metric)
    df_piv.columns = [f'Janela {j}' for j in df_piv.columns]
    sns.heatmap(
        df_piv, ax=ax, annot=True, fmt='.3f',
        cmap='YlOrRd', linewidths=0.5, cbar_kws={'shrink': 0.7}
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
records_excl = []
for janela in JANELAS:
    cols_pred = [f'pred_{m}' for m in MODELOS]
    df_j = df_all[df_all['Janela Incremental'] == janela].dropna(subset=cols_pred)

    for m, label in zip(MODELOS, LABELS):
        acertou      = df_j[f'pred_{m}'] == df_j[TARGET]
        outros_erram = np.all(
            [df_j[f'pred_{om}'] != df_j[TARGET] for om in MODELOS if om != m],
            axis=0
        )
        excl = int((acertou & outros_erram).sum())
        total_acertos = int(acertou.sum())
        records_excl.append({
            'Janela': f'Janela {janela}',
            'Modelo': label,
            'Acertos Exclusivos': excl,
            'Total Acertos': total_acertos,
            '% Exclusivos': excl / total_acertos if total_acertos > 0 else 0
        })

df_excl = pd.DataFrame(records_excl)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.barplot(
    data=df_excl, x='Modelo', y='Acertos Exclusivos',
    hue='Janela', palette='tab10', ax=axes[0]
)
axes[0].set_title('Acertos Exclusivos por Modelo (n° absoluto)', fontsize=12)
axes[0].set_xlabel('')

sns.barplot(
    data=df_excl, x='Modelo', y='% Exclusivos',
    hue='Janela', palette='tab10', ax=axes[1]
)
axes[1].set_title('Acertos Exclusivos (% sobre total de acertos do modelo)', fontsize=12)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

## 2. Calibração das probabilidades

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(22, 12))

for row, janela in enumerate(JANELAS):
    df_j = df_all[df_all['Janela Incremental'] == janela]

    for col, (m, label, color) in enumerate(zip(MODELOS, LABELS, PALETTE)):
        ax = axes[row][col]
        subset = df_j[[f'prob_{m}', TARGET]].dropna()

        frac_pos, mean_pred = calibration_curve(
            subset[TARGET], subset[f'prob_{m}'],
            n_bins=10, strategy='uniform'
        )

        ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1, label='Perfeito')
        ax.plot(mean_pred, frac_pos, marker='o', color=color, linewidth=2, markersize=5, label=label)
        ax.fill_between(mean_pred, mean_pred, frac_pos, alpha=0.15, color=color)

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(f'{label} | Janela {janela}', fontsize=9)
        ax.set_xlabel('Prob. prevista', fontsize=8)
        ax.set_ylabel('Fração positivos', fontsize=8)
        ax.tick_params(labelsize=7)

fig.suptitle('Curvas de Calibração por Modelo e Janela', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
records_pp = []
for janela in JANELAS:
    df_j  = df_all[df_all['Janela Incremental'] == janela]
    df_js = df_j.dropna(subset=['pred_stack'])

    records_pp.append({'Janela': f'Janela {janela}', 'Tipo': 'Real', 'Taxa': df_js[TARGET].mean()})
    records_pp.append({'Janela': f'Janela {janela}', 'Tipo': 'Stack', 'Taxa': (df_js['pred_stack'] == 1).mean()})

    for m, label in zip(MODELOS, LABELS):
        df_jm = df_j.dropna(subset=[f'pred_{m}'])
        records_pp.append({'Janela': f'Janela {janela}', 'Tipo': label, 'Taxa': (df_jm[f'pred_{m}'] == 1).mean()})

df_pp = pd.DataFrame(records_pp)

fig, ax = plt.subplots(figsize=(15, 5))
ordem_tipos = ['Real', 'Stack'] + LABELS
cores_bar   = ['#2d6a4f', STACK_COR] + list(PALETTE)

sns.barplot(
    data=df_pp, x='Janela', y='Taxa', hue='Tipo',
    hue_order=ordem_tipos, palette=cores_bar, ax=ax
)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('Taxa de Predição Classe 1 vs. Frequência Real', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Proporção prevista como Classe 1')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
records_cal = []
for janela in JANELAS:
    df_j = df_all[df_all['Janela Incremental'] == janela]

    for m, label in zip(MODELOS, LABELS):
        s = df_j[[f'prob_{m}', TARGET]].dropna()
        records_cal.append({
            'Janela': f'Janela {janela}', 'Modelo': label,
            'Brier': brier_score_loss(s[TARGET], s[f'prob_{m}']),
            'Log-Loss': log_loss(s[TARGET], s[f'prob_{m}'])
        })

    s_stk = df_j[['prob_stack', TARGET]].dropna()
    records_cal.append({
        'Janela': f'Janela {janela}', 'Modelo': 'Stack',
        'Brier': brier_score_loss(s_stk[TARGET], s_stk['prob_stack']),
        'Log-Loss': log_loss(s_stk[TARGET], s_stk['prob_stack'])
    })

df_cal = pd.DataFrame(records_cal)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ordem_modelos = LABELS + ['Stack']
cores_modelos = list(PALETTE) + [STACK_COR]

for ax, metric, title in zip(
    axes,
    ['Brier', 'Log-Loss'],
    ['Brier Score  (↓ melhor)', 'Log-Loss  (↓ melhor)']
):
    sns.barplot(
        data=df_cal, x='Modelo', y=metric,
        hue='Janela', palette='Set2', ax=ax,
        order=ordem_modelos
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(22, 10))

for row, janela in enumerate(JANELAS):
    df_j = df_all[df_all['Janela Incremental'] == janela]

    for col, (m, label, color) in enumerate(zip(MODELOS, LABELS, PALETTE)):
        ax = axes[row][col]
        probs = df_j[f'prob_{m}'].dropna()

        sns.histplot(
            probs, bins=25, color=color, ax=ax,
            stat='density', kde=True, kde_kws={'bw_adjust': 1.2}
        )
        ax.axvline(0.5, color='black', linestyle='--', alpha=0.5, linewidth=1)
        ax.set_title(f'{label} | Janela {janela}', fontsize=9)
        ax.set_xlabel('Probabilidade', fontsize=8)
        ax.set_ylabel('Densidade', fontsize=8)
        ax.set_xlim(0, 1)
        ax.tick_params(labelsize=7)

fig.suptitle('Distribuição das Probabilidades por Modelo e Janela', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
thresholds = np.arange(0.30, 0.71, 0.01)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, janela in zip(axes, JANELAS):
    df_j = df_all[df_all['Janela Incremental'] == janela].dropna(subset=['prob_stack', TARGET])

    accs = [
        accuracy_score(df_j[TARGET], (df_j['prob_stack'] >= t).astype(int))
        for t in thresholds
    ]

    melhor_t   = thresholds[int(np.argmax(accs))]
    melhor_acc = max(accs)

    ax.plot(thresholds, accs, color=STACK_COR, linewidth=2.5)
    ax.axvline(melhor_t,  color='crimson', linestyle='--', linewidth=1.5,
               label=f'Ótimo: {melhor_t:.2f}  ({melhor_acc:.3f})')
    ax.axvline(0.50, color='gray', linestyle=':', linewidth=1.2, alpha=0.8,
               label='Limiar = 0.50')

    acc_padrao = accuracy_score(df_j[TARGET], (df_j['prob_stack'] >= 0.5).astype(int))
    ax.scatter([0.5], [acc_padrao], color='gray', s=60, zorder=5)

    ax.set_title(f'Janela = {janela}', fontsize=12)
    ax.set_xlabel('Threshold de decisão')
    ax.set_ylabel('Acurácia')
    ax.set_ylim(0.55, 0.75)
    ax.legend(fontsize=9)

fig.suptitle('Acurácia do Stack por Limiar de Decisão', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Comportamento ao longo da temporada

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, janela in zip(axes, JANELAS):
    df_j = df_all[df_all['Janela Incremental'] == janela]

    for m, label, color in zip(MODELOS, LABELS, PALETTE):
        df_m = df_j.dropna(subset=[f'pred_{m}'])
        acc_pos = (
            df_m.groupby('Ordem Jogo Temporada')
            .apply(lambda g: (g[f'pred_{m}'] == g[TARGET]).mean())
            .reset_index(name='acc')
            .sort_values('Ordem Jogo Temporada')
        )
        rolling = acc_pos['acc'].rolling(10, min_periods=3).mean()
        ax.plot(
            acc_pos['Ordem Jogo Temporada'], rolling,
            color=color, linewidth=1.2, alpha=0.5, label=label
        )

    df_stk = df_j.dropna(subset=['pred_stack'])
    acc_stk = (
        df_stk.groupby('Ordem Jogo Temporada')
        .apply(lambda g: (g['pred_stack'] == g[TARGET]).mean())
        .reset_index(name='acc')
        .sort_values('Ordem Jogo Temporada')
    )
    rolling_stk = acc_stk['acc'].rolling(10, min_periods=3).mean()
    ax.plot(
        acc_stk['Ordem Jogo Temporada'], rolling_stk,
        color=STACK_COR, linewidth=2.5, label='Stack', zorder=5
    )

    ax.set_title(f'Janela = {janela}', fontsize=12)
    ax.set_xlabel('Posição na temporada')
    ax.set_ylabel('Acurácia (rolling 10)')
    ax.set_ylim(0.40, 0.90)
    ax.legend(fontsize=8, ncol=2)

fig.suptitle('Acurácia por Posição na Temporada (média móvel de 10 jogos)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, janela in zip(axes, JANELAS):
    df_j = df_all[df_all['Janela Incremental'] == janela].dropna(subset=['pred_stack'])

    acc_pos = (
        df_j.groupby('Ordem Jogo Temporada')
        .apply(lambda g: (g['pred_stack'] == g[TARGET]).mean())
        .reset_index(name='acc')
        .sort_values('Ordem Jogo Temporada')
    )
    acc_pos['acc_cum'] = acc_pos['acc'].expanding().mean()

    ax.plot(
        acc_pos['Ordem Jogo Temporada'], acc_pos['acc_cum'],
        color=STACK_COR, linewidth=2.5
    )
    ax.axhline(
        acc_pos['acc'].mean(), color='gray',
        linestyle='--', alpha=0.7, linewidth=1.5, label='Média geral'
    )

    ax.set_title(f'Janela = {janela}', fontsize=12)
    ax.set_xlabel('Posição na temporada')
    ax.set_ylabel('Acurácia acumulada')
    ax.set_ylim(0.40, 0.90)
    ax.legend(fontsize=9)

fig.suptitle('Acurácia Acumulada do Stack ao Longo da Temporada', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
CORTE = 20
records_fase = []

for janela in JANELAS:
    df_j = df_all[df_all['Janela Incremental'] == janela]
    limite_inicio = janela + CORTE

    for m, label in zip(MODELOS + ['stack'], LABELS + ['Stack']):
        pred_col = f'pred_{m}' if m != 'stack' else 'pred_stack'

        for fase, mask in [
            (f'Primeiros {CORTE} jogos', df_j['Ordem Jogo Temporada'] <= limite_inicio),
            ('Demais jogos',             df_j['Ordem Jogo Temporada'] >  limite_inicio)
        ]:
            s = df_j.loc[mask].dropna(subset=[pred_col])
            if len(s) == 0:
                continue
            acc = accuracy_score(s[TARGET], s[pred_col])
            records_fase.append({
                'Janela': f'Janela {janela}',
                'Modelo': label,
                'Fase': fase,
                'Acurácia': round(acc, 4),
                'N': len(s)
            })

df_fase = pd.DataFrame(records_fase)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
ordem_modelos = LABELS + ['Stack']
cores_fase    = sns.color_palette('Set2', 2)

for ax, janela in zip(axes, JANELAS):
    sub = df_fase[df_fase['Janela'] == f'Janela {janela}']
    sns.barplot(
        data=sub, x='Modelo', y='Acurácia',
        hue='Fase', palette=cores_fase,
        order=ordem_modelos, ax=ax
    )
    ax.set_title(f'Janela = {janela}', fontsize=12)
    ax.set_xlabel('')
    ax.set_ylim(0.40, 0.85)
    ax.tick_params(axis='x', rotation=20)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))

fig.suptitle(f'Acurácia: Primeiros {CORTE} jogos da temporada vs. Demais', fontsize=14)
plt.tight_layout()
plt.show()

tabela_pivot = df_fase.pivot_table(
    index=['Janela', 'Modelo'], columns='Fase',
    values='Acurácia', aggfunc='first'
).reset_index()

col_inicio = f'Primeiros {CORTE} jogos'
tabela_pivot['Diferença'] = (tabela_pivot['Demais jogos'] - tabela_pivot[col_inicio]).round(4)
print(tabela_pivot.to_string(index=False))

## Evolução dos coeficientes da regressão logística

> Esta célula re-treina o meta-modelo incrementalmente para extrair os coeficientes .

In [ ]:
coef_records = []

for janela in JANELAS:
    df_j = df_meta[df_meta['Janela Incremental'] == janela].copy()

    for temporada in sorted(df_j['Temporada'].unique()):
        df_t = (
            df_j[df_j['Temporada'] == temporada]
            .sort_values('Ordem Jogo Temporada')
            .reset_index(drop=True)
        )
        n = len(df_t)

        for i in range(janela, n):
            X_tr = df_t.loc[:i - 1, FEATURES]
            y_tr = df_t.loc[:i - 1, TARGET]

            if y_tr.nunique() < 2:
                continue

            model = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
            model.fit(X_tr, y_tr)

            for feat, coef in zip(LABELS, model.coef_[0]):
                coef_records.append({
                    'Janela':    janela,
                    'Temporada': temporada,
                    'N_treino':  i,
                    'Feature':   feat,
                    'Coef':      coef
                })

df_coef = pd.DataFrame(coef_records)

df_coef_med = (
    df_coef
    .groupby(['Janela', 'N_treino', 'Feature'])['Coef']
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, janela in zip(axes, JANELAS):
    sub = df_coef_med[df_coef_med['Janela'] == janela]

    for feat, color in zip(LABELS, PALETTE):
        s = sub[sub['Feature'] == feat].sort_values('N_treino')
        smooth = s.set_index('N_treino')['Coef'].rolling(15, min_periods=5).mean()
        ax.plot(smooth.index, smooth.values, label=feat, color=color, linewidth=1.8)

    ax.axhline(0, color='black', linestyle='--', alpha=0.4, linewidth=1)
    ax.set_title(f'Janela = {janela}', fontsize=12)
    ax.set_xlabel('N° de exemplos de treino acumulados')
    ax.set_ylabel('Coeficiente médio')
    ax.legend(fontsize=8)

fig.suptitle('Evolução dos Coeficientes da Regressão Logística (média entre temporadas)', fontsize=14)
plt.tight_layout()
plt.show()